# Credit Card Fraud Detection — DL Homework

Бінарна класифікація на сильно незбалансованому датасеті
[`mlg-ulb/creditcardfraud`](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud).

**План:**
1. Завантажити дані, EDA (дисбаланс класів).
2. Препроцесинг: StandardScaler + stratified split на train/val/test.
3. Порахувати `class_weight` для боротьби з дисбалансом.
4. Дві архітектури:
   - Проста: `Dense(32) → Dense(16) → Dense(1)`
   - Глибша: `Dense(256)+BN+ReLU+Dropout → Dense(128)+BN+ReLU+Dropout → Dense(64)+BN+ReLU+Dropout → Dense(1)`
5. Callbacks: `EarlyStopping(monitor="val_precision", mode="max", patience=15)`,
   `ReduceLROnPlateau`, `LearningRateScheduler`, `TensorBoard`, кастомний (без Telegram).
6. Метрики: confusion matrix, precision, recall, F1, ROC-AUC.
7. Порівняння при різних порогах (0.3 / 0.5 / 0.8).

> Для fraud-детекції accuracy безглузда (≈99.8% досягається моделлю, що завжди передбачає "не шахрайство").
> Дивимось на **recall** (скільки шахрайств виявили) та **precision/F1** на меншому класі.


## 1. Імпорти

In [ ]:
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
)

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))


## 2. Завантаження даних

У Colab найзручніше через `kagglehub` — він кешує датасет.

In [ ]:
# Якщо в Colab — встанови kagglehub один раз
# !pip install -q kagglehub

import kagglehub

path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
df = pd.read_csv(f"{path}/creditcard.csv")
print(df.shape)
df.head()


## 3. EDA — дисбаланс класів

In [ ]:
counts = df["Class"].value_counts()
ratio = df["Class"].mean()
print(counts)
print(f"Fraud rate: {ratio * 100:.4f}%  (1 шахрайство на ~{int(1/ratio)} транзакцій)")

fig, ax = plt.subplots(figsize=(5, 3))
counts.plot(kind="bar", ax=ax)
ax.set_title("Class distribution (0 = legit, 1 = fraud)")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()


## 4. Препроцесинг

- Stratified split 70 / 15 / 15 (train / val / test).
- `StandardScaler` фітимо **тільки** на train, щоб не було leakage.

In [ ]:
X = df.drop("Class", axis=1).values
y = df["Class"].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, frauds = {int(y_train.sum())}")
print(f"Val:   {X_val.shape}, frauds = {int(y_val.sum())}")
print(f"Test:  {X_test.shape}, frauds = {int(y_test.sum())}")


## 5. Class weights

`compute_class_weight('balanced', ...)` дає ваги обернено пропорційні частоті:
`w_i = n_samples / (n_classes * n_i)`. Це штрафує модель сильніше за помилки на меншому класі.

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = dict(zip(classes.tolist(), weights.tolist()))
print("class_weight:", class_weight)


## 6. Кастомний Callback (без Telegram)

Просто друкує повідомлення, коли val_precision переходить заданий поріг —
аналог того, що було в умові, лише без HTTP-запиту.

In [ ]:
class HighPrecisionAlert(callbacks.Callback):
    def __init__(self, threshold: float = 0.9):
        super().__init__()
        self.threshold = threshold
        self.triggered = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_prec = logs.get("val_precision")
        if val_prec is not None and val_prec > self.threshold and not self.triggered:
            print(
                f"\n[Alert] Epoch {epoch + 1}: val_precision = {val_prec:.4f} "
                f"перевищив поріг {self.threshold}"
            )
            self.triggered = True


## 7. Збірка callbacks

- `EarlyStopping(monitor="val_precision", mode="max", patience=15)` —
  завдання просить моніторити precision або recall з `mode="max"`.
- `ReduceLROnPlateau` — знижує LR коли val_loss перестає падати.
- `LearningRateScheduler` — плавне експоненційне зменшення після 10-ї епохи.
  > Зверни увагу: ці два callbacks конфліктують, бо обидва міняють LR.
  > Тут вмикаю один з двох через прапор.

In [ ]:
LOG_ROOT = "logs"


def build_callbacks(name: str, use_scheduler: bool = False) -> list[callbacks.Callback]:
    log_dir = f"{LOG_ROOT}/{name}_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    cbs: list[callbacks.Callback] = [
        callbacks.EarlyStopping(
            monitor="val_precision",
            mode="max",
            patience=15,
            restore_best_weights=True,
            verbose=1,
        ),
        callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1),
        HighPrecisionAlert(threshold=0.9),
    ]
    if use_scheduler:
        cbs.append(
            callbacks.LearningRateScheduler(
                lambda epoch, lr: float(lr * 0.95) if epoch > 10 else float(lr),
                verbose=0,
            )
        )
    else:
        cbs.append(
            callbacks.ReduceLROnPlateau(
                monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
            )
        )
    return cbs


## 8. Модель 1 — проста (32 → 16 → 1)

In [ ]:
def build_simple_model(input_dim: int) -> keras.Model:
    model = keras.Sequential(
        [
            keras.Input(shape=(input_dim,)),
            layers.Dense(32, activation="relu"),
            layers.Dense(16, activation="relu"),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="simple_32_16",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


simple_model = build_simple_model(X_train.shape[1])
simple_model.summary()


In [ ]:
history_simple = simple_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=2048,
    class_weight=class_weight,
    callbacks=build_callbacks("simple", use_scheduler=False),
    verbose=2,
)


## 9. Модель 2 — глибша з BN + Dropout

In [ ]:
def build_deep_model(input_dim: int, dropout: float = 0.3) -> keras.Model:
    model = keras.Sequential(
        [
            keras.Input(shape=(input_dim,)),
            layers.Dense(256),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout),
            layers.Dense(128),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout),
            layers.Dense(64),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout),
            layers.Dense(1, activation="sigmoid"),
        ],
        name="deep_256_128_64",
    )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )
    return model


deep_model = build_deep_model(X_train.shape[1])
deep_model.summary()


In [ ]:
history_deep = deep_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=2048,
    class_weight=class_weight,
    callbacks=build_callbacks("deep", use_scheduler=True),
    verbose=2,
)


## 10. Криві навчання

In [ ]:
def plot_history(history, title: str):
    h = history.history
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(h["loss"], label="train")
    axes[0].plot(h["val_loss"], label="val")
    axes[0].set_title(f"{title} — loss")
    axes[0].legend()
    axes[1].plot(h["precision"], label="train precision")
    axes[1].plot(h["val_precision"], label="val precision")
    axes[1].plot(h["recall"], label="train recall", linestyle="--")
    axes[1].plot(h["val_recall"], label="val recall", linestyle="--")
    axes[1].set_title(f"{title} — precision / recall")
    axes[1].legend()
    axes[2].plot(h["auc"], label="train")
    axes[2].plot(h["val_auc"], label="val")
    axes[2].set_title(f"{title} — ROC-AUC")
    axes[2].legend()
    for ax in axes:
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_history(history_simple, "simple")
plot_history(history_deep, "deep")


## 11. Оцінка на test + експерименти з порогом

`precision_score`, `recall_score`, `f1_score`, `roc_auc_score` + confusion matrix.
Порівнюємо пороги 0.3 / 0.5 / 0.8 — як зсувається precision/recall trade-off.

In [ ]:
def evaluate_at_threshold(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    threshold: float = 0.5,
    name: str = "",
) -> dict:
    probs = model.predict(X, verbose=0).flatten()
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(y, preds)
    prec = precision_score(y, preds, zero_division=0)
    rec = recall_score(y, preds, zero_division=0)
    f1 = f1_score(y, preds, zero_division=0)
    auc = roc_auc_score(y, probs)
    return {
        "model": name,
        "threshold": threshold,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
        "tn": cm[0, 0],
        "fp": cm[0, 1],
        "fn": cm[1, 0],
        "tp": cm[1, 1],
    }


rows = []
for model, name in [(simple_model, "simple"), (deep_model, "deep")]:
    for t in [0.3, 0.5, 0.8]:
        rows.append(evaluate_at_threshold(model, X_test, y_test, threshold=t, name=name))

results = pd.DataFrame(rows)
results


In [ ]:
# Confusion matrices при порозі 0.5
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (model, name) in zip(axes, [(simple_model, "simple"), (deep_model, "deep")]):
    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False)
    ax.set_title(f"{name} @ 0.5")
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.show()


## 12. PR- і ROC-криві

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model, name in [(simple_model, "simple"), (deep_model, "deep")]:
    probs = model.predict(X_test, verbose=0).flatten()
    p, r, _ = precision_recall_curve(y_test, probs)
    fpr, tpr, _ = roc_curve(y_test, probs)
    axes[0].plot(r, p, label=name)
    axes[1].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, probs):.4f})")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set(xlabel="FPR", ylabel="TPR", title="ROC")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 13. TensorBoard

Запусти комірку нижче — і прямо тут з'явиться інтерактивний дашборд з графіками loss / precision / recall / AUC для обох моделей.

Кожен run (`simple_...`, `deep_...`) — окрема підпапка в `logs/`, їх можна вмикати/вимикати чекбоксами зліва і порівнювати на одному графіку.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

## 14. Висновки

- **Accuracy безглузда** для цього датасету — клас 1 ≈ 0.17%. Використовуємо precision / recall / F1 / ROC-AUC.
- **Яку помилку зменшувати?** Для fraud-детекції зазвичай критичніший **False Negative**
  (пропущене шахрайство), тому моніторимо `recall` або `F1` як основну метрику —
  тут моніторимо `val_precision` лише як приклад з умови.
- **class_weight** + **знижений поріг (0.3)** дають вищий recall, але страждає precision (більше FP).
  **Поріг 0.8** навпаки — менше алертів, але можна пропустити шахрайство. Бізнес обирає компроміс.
- Глибша модель з BN + Dropout стабільніше тренується і зазвичай дає трохи кращий ROC-AUC,
  але на цьому датасеті проста модель теж сильна — фічі вже PCA-трансформовані.